# 10年定着予測 — CPUモデル動物園 × 事前登録ビュー（`71_`）

## 問題意識

`68_` で **「モデル × 特徴量ビュー」は2次元**であることが判明した（TabPFN は full441 で 0.519 →
`hire_fixed`(78列) で 0.4838、**ビューだけで 0.035** 動いた）。
ところが **`68_` 以前のモデル却下は、すべて 441列という単一ビューで下されている**。

| ノートブック | 却下したモデル | 使ったビュー |
|---|---|---|
| `35_` / `43_` | LightGBM・XGBoost | 441列 / 113列 |
| `36_` | **NN(MLP+Embedding)・SVM(RBF)** | **441列のみ** |

SVM(RBF) と MLP は高次元・疎な441列（TF-IDF SVD込み）で最も不利になるモデルであり、
**低次元ビューで再評価すれば結論が変わりうる**。本ノートブックはそれを CPU だけで検証する。

## 事前登録（実行前に確定。結果を見てから変えない）

### 1. ビューはモデルごとに「機構」から1つだけ割り当てる。5択を回して val で選ばない

`39_`（4/4ブートストラップCIを通ったのに Public で符号反転）・`42_`（LOO診断で選んだら Public +0.0098）で
繰り返し焼かれた通り、**複数構成を val で比較して勝者を選ぶと勝者の呪いが働く**
（[[ablation-cannot-settle-feature-blocks]]）。そこで各モデルに**1ビューだけ**を理屈で割り当てる。

| モデル | ビュー | 事前の理屈 |
|---|---|---|
| CatBoost（基準） | `full441` | ordered target statistics が多カテゴリに強い（`43_`で実測済み） |
| **EBM (GA2M)** | `cb_top150` | 加法＋2次交互作用のみ。441列だとペア数が爆発し正則化が効かない |
| **Gaussian Process** | `num_top50_std` | カーネル法は次元の呪いに弱い。n=2208 なら厳密GPが可能 |
| **SVM (RBF)** | `num_top50_std` | 同上。`36_`が441列で却下したのは条件が悪すぎた |
| **MLP** | `hire_fixed`(78) | 同上。小データMLPにTF-IDF SVD込み441列は最悪の条件 |
| **ExtraTrees** | `full441` | 木なので高次元に耐える。相関を下げる要員 |
| **Logistic(L2)** | `hire_fixed`(78) | `31_`は441列で試した。低次元なら化ける可能性 |

### 2. ハイパーパラメータ探索は「学習データ内の内側CV」でのみ行う

Optuna は **`ag_train_80b`(2208名) の内側3-fold CV** だけを見る。
検証セット535名は**一度も探索に使わない**ので、ゲート判定が探索で汚染されない。
（CatBoost の再探索は [[hyperparameter-retuning-exhausted]] の通り打ち止めなので行わない。
新しいモデル族には未実施なので実施する。）

### 3. 採否ゲート（ラベルを見る指標は足切りにのみ使う）

- **(A) 足切り**: 単体 val ≤ CatBoost val + `GATE_VAL_MARGIN`(0.02)
  — [[validation-asymmetry]] の通り val は「悪化」と言うときだけ信用できる
- **(B) 多様性**: CatBoost val予測との相関 < `GATE_CORR_MAX`(0.96)
  — `46_` の教訓。等重み平均の可否は強さだけでなく**相関**で決まる

**(A) かつ (B) を満たしたものだけをアンサンブル候補**とする。
val スコアの順位付けは行わない。**本ノートブックでは提出ファイルを作らない**
（ブレンドと提出は結果を見てから別途）。

---

## ⚠️ 本ノートブックが再利用する「過去の実行結果」（新規に計算しないもの）

**以下は本ノートブックでは学習し直さず、保存済みファイルを読み込むだけである。**
読み込み時にログへ `【再利用】` と出力し、結果表でも `source` 列で区別する。

| 用途 | 読み込むファイル | 生成元 | 注意点 |
|---|---|---|---|
| 相関行列の比較対象（val） | `..._68_foundation_models_3way_{tabpfn,tabicl,tabdpt}_hire_fixed_valpreds.npy` | **`68_`** | **`68_`は速度優先の縮小設定**（TabPFN `n_estimators=4`、TabICL `batch_size=1`）。本番設定(`63_`既定=8)ではないので、絶対値の比較には使わず**相関の把握にのみ使う** |
| 相関行列の比較対象（test） | `..._70_hire_fixed_fm_submission_{tabpfn,tabicl,tabdpt}_hire_fixed_testpreds.npy` | **`70_`** | こちらは本番設定。Public 0.511832 を出した構成の素材 |
| AutoGluonプールとの相関 | `20260816_pool_poolC_weighted.csv` | **`62_`までの8実行の平均**（`50_`/`51_`/`53_`/`61_`×2/`62_`×4 の full441 weighted） | `56_`のLM込み444列は特徴量が異質なため除外済み。`64_`で追加中のシードは**まだ含まれていない** |
| CatBoost基準の突き合わせ（任意） | `..._54_l2_m_interaction_R0_memofix_plus_LM_valpreds.npy` | **`54_`** | 単層CatBoostの最良(Public 0.515030)。本NBは CatBoost を再学習するので**参考照合のみ** |

**新規に学習するのは EBM / GP / SVM / MLP / ExtraTrees / Logistic の6つと、基準の CatBoost だけ**である。
TabPFN・TabICL・TabDPT は **GPU が必要なため本ノートブックでは一切学習しない**。

さらに、本ノートブック自身の計算結果も `data/output/<TODAY>/` に `.npy` でキャッシュする。
2回目以降の実行では**キャッシュを読むだけになる**ので、そのときも `【再利用】` とログに出る。
やり直したいときは `RECOMPUTE_ALL = True` にすること。

> ⚠️ [[checkpoint-drive-sync-gotcha]]: ローカルMacとColabは同じDriveを見ている。
> Colabで独立に回したいのに一瞬で終わった場合は、キャッシュを読んでいる。

In [1]:
# EBM(interpret) だけ追加インストールが必要。他は catboost / scikit-learn / optuna で足りる。
# ⚠️ GPUは使わない。TabPFN/TabICL/TabDPT は本NBでは学習しないのでインストール不要。
!pip install -q interpret==0.6.* catboost optuna


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 11.9 MB/s eta 0:00:0000:010:01
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.2/47.2 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 12.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 21.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.9/265.9 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 72.8 MB/s eta 0:00:00 0:00:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 84.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 72.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.1/780.1 kB 41.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.7/101.7 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import multiprocessing
print(f"CPUコア数: {multiprocessing.cpu_count()}（今回はCPUで学習するため、GPUランタイムは不要）")

CPUコア数: 8（今回はCPUで学習するため、GPUランタイムは不要）


In [3]:
import sys
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = Path("/content/drive/MyDrive/jaggle_2026")
sys.path.append(str(PROJECT_ROOT))


Mounted at /content/drive


In [4]:
import datetime
import json
import re
import warnings

import numpy as np
import pandas as pd
import catboost as cb
import optuna
from scipy import stats
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import log_loss
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler

from common.utils.logger import get_logger
from common.utils.metrics import calculate_logloss
from common.utils.seed import seed_everything

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED = 42
seed_everything(seed=SEED)

TARGET_COL = "10年定着ラベル"
ID_COL = "社員ID"

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)


In [5]:
SCRIPT_NAME = "71_model_zoo_cpu"
TODAY = datetime.datetime.now().strftime("%Y%m%d")

LOG_DIR = PROJECT_ROOT / "logs"
logger = get_logger(SCRIPT_NAME, log_dir=str(LOG_DIR))
logger.info(f"=== [{SCRIPT_NAME}] 実験開始 ===")

OUTPUT_DIR = PROJECT_ROOT / "data" / "output" / TODAY
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SAVED_MODELS_DIR = PROJECT_ROOT / "saved_models" / TODAY / SCRIPT_NAME
SAVED_MODELS_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_DIR = PROJECT_ROOT / "data" / "output" / "_checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = CHECKPOINT_DIR / f"{SCRIPT_NAME}_checkpoint.csv"

RESET_CHECKPOINT = False  # Trueにすると既存チェックポイントを削除して最初から再計算する
if RESET_CHECKPOINT and CHECKPOINT_PATH.exists():
    CHECKPOINT_PATH.unlink()
    print("チェックポイントを削除しました（全構成を再計算します）")

logger.info(f"Output Directory: {OUTPUT_DIR}")
logger.info(f"Checkpoint Path: {CHECKPOINT_PATH}")
if CHECKPOINT_PATH.exists():
    logger.info(f"既存のチェックポイントを発見: {len(pd.read_csv(CHECKPOINT_PATH))}件の結果が記録済み")
else:
    logger.info("チェックポイントは未作成（新規実行）")



[2026-08-16 13:51:16] [INFO] === [71_model_zoo_cpu] 実験開始 ===


INFO:71_model_zoo_cpu:=== [71_model_zoo_cpu] 実験開始 ===


[2026-08-16 13:51:17] [INFO] Output Directory: /content/drive/MyDrive/jaggle_2026/data/output/20260816


INFO:71_model_zoo_cpu:Output Directory: /content/drive/MyDrive/jaggle_2026/data/output/20260816


[2026-08-16 13:51:17] [INFO] Checkpoint Path: /content/drive/MyDrive/jaggle_2026/data/output/_checkpoints/71_model_zoo_cpu_checkpoint.csv


INFO:71_model_zoo_cpu:Checkpoint Path: /content/drive/MyDrive/jaggle_2026/data/output/_checkpoints/71_model_zoo_cpu_checkpoint.csv


[2026-08-16 13:51:18] [INFO] チェックポイントは未作成（新規実行）


INFO:71_model_zoo_cpu:チェックポイントは未作成（新規実行）


In [6]:
INPUT_DIR = PROJECT_ROOT / "data" / "input"

train_persona = pd.read_csv(INPUT_DIR / "employee_persona_train.csv")
test_persona = pd.read_csv(INPUT_DIR / "employee_persona_test.csv")
train_monthly = pd.read_csv(INPUT_DIR / "employee_monthly_train.csv")
test_monthly = pd.read_csv(INPUT_DIR / "employee_monthly_test.csv")

logger.info(f"Train Persona Shape: {train_persona.shape}, Test Persona Shape: {test_persona.shape}")
logger.info(f"Train Monthly Shape: {train_monthly.shape}, Test Monthly Shape: {test_monthly.shape}")

y_train = train_persona[TARGET_COL]
train_ids = train_persona[ID_COL].values
test_ids = test_persona[ID_COL].values

logger.info(f"定着率: {y_train.mean():.4f}")
logger.info(f"Train IDs: {len(train_ids)}, Test IDs: {len(test_ids)}")


[2026-08-16 13:51:22] [INFO] Train Persona Shape: (2761, 20), Test Persona Shape: (2502, 19)


INFO:71_model_zoo_cpu:Train Persona Shape: (2761, 20), Test Persona Shape: (2502, 19)


[2026-08-16 13:51:22] [INFO] Train Monthly Shape: (65754, 29), Test Monthly Shape: (60048, 29)


INFO:71_model_zoo_cpu:Train Monthly Shape: (65754, 29), Test Monthly Shape: (60048, 29)


[2026-08-16 13:51:22] [INFO] 定着率: 0.5647


INFO:71_model_zoo_cpu:定着率: 0.5647


[2026-08-16 13:51:22] [INFO] Train IDs: 2761, Test IDs: 2502


INFO:71_model_zoo_cpu:Train IDs: 2761, Test IDs: 2502


## 0. 早期退職者の特定

`月末在籍状態 == "退職"` の行を持つ社員が、0-23ヶ月の観測期間中に退職した社員。
Train に129名（全員ラベル0）、Test には0名（[[test-set-is-survivor-filtered]]）。


In [7]:
EARLY_LEAVER_IDS = set(train_monthly.loc[train_monthly["月末在籍状態"] == "退職", ID_COL].unique())
_test_early = set(test_monthly.loc[test_monthly["月末在籍状態"] == "退職", ID_COL].unique())

logger.info(f"Train 早期退職者: {len(EARLY_LEAVER_IDS)}名 / {len(train_ids)}名 ({len(EARLY_LEAVER_IDS)/len(train_ids):.1%})")
logger.info(f"Test  早期退職者: {len(_test_early)}名 / {len(test_ids)}名")
_y_idx = train_persona.set_index(ID_COL)[TARGET_COL]
logger.info(f"早期退職者のラベル平均: {_y_idx.loc[list(EARLY_LEAVER_IDS)].mean():.4f}（0.0のはず）")
logger.info(f"定着率: 全体 {y_train.mean():.4f} / 早期退職者を除く {_y_idx[~_y_idx.index.isin(EARLY_LEAVER_IDS)].mean():.4f}")
assert len(_test_early) == 0, "Testに早期退職者が存在する。前提が崩れているので調査すること"


[2026-08-16 13:51:22] [INFO] Train 早期退職者: 129名 / 2761名 (4.7%)


INFO:71_model_zoo_cpu:Train 早期退職者: 129名 / 2761名 (4.7%)


[2026-08-16 13:51:22] [INFO] Test  早期退職者: 0名 / 2502名


INFO:71_model_zoo_cpu:Test  早期退職者: 0名 / 2502名


[2026-08-16 13:51:22] [INFO] 早期退職者のラベル平均: 0.0000（0.0のはず）


INFO:71_model_zoo_cpu:早期退職者のラベル平均: 0.0000（0.0のはず）


[2026-08-16 13:51:22] [INFO] 定着率: 全体 0.5647 / 早期退職者を除く 0.5923


INFO:71_model_zoo_cpu:定着率: 全体 0.5647 / 早期退職者を除く 0.5923


## 1. 基本特徴量関数の定義（split非依存、`51_`と同一ロジック）

In [8]:
def create_monthly_aggregation_features(monthly_df, employee_ids):
    """月次データから集約特徴量を生成（12_〜18_と同一ロジック）"""
    numeric_cols = [
        "残業時間", "有給取得日数", "欠勤日数", "研修時間",
        "上司との面談実施回数", "情報共有件数", "在宅勤務日数",
        "360度評価_親和度", "360度評価_信頼度", "360度評価_主体度",
        "360度評価_学習度", "360度評価_共有貢献度", "360度評価者数",
        "顧客満足度評価", "担当プロジェクト数", "月例給与_円"
    ]

    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].copy()
        emp_data = emp_data.sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}

        for col in numeric_cols:
            if col not in emp_data.columns:
                continue
            values = emp_data[col].values
            valid_values = values[~pd.isna(values)]

            features[f"{col}_mean"] = np.mean(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_std"] = np.std(valid_values) if len(valid_values) > 1 else np.nan
            features[f"{col}_min"] = np.min(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_max"] = np.max(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_median"] = np.median(valid_values) if len(valid_values) > 0 else np.nan
            mean_val = features[f"{col}_mean"]
            std_val = features[f"{col}_std"]
            features[f"{col}_cv"] = std_val / mean_val if (mean_val and mean_val != 0) else np.nan

            early = emp_data[emp_data["経過月数"].between(0, 2)][col]
            mid = emp_data[emp_data["経過月数"].between(3, 11)][col]
            late = emp_data[emp_data["経過月数"].between(12, 23)][col]
            features[f"{col}_early_mean"] = early.mean()
            features[f"{col}_mid_mean"] = mid.mean()
            features[f"{col}_late_mean"] = late.mean()
            features[f"{col}_late_minus_early"] = late.mean() - early.mean()
            features[f"{col}_late_early_ratio"] = (
                late.mean() / early.mean() if early.mean() and early.mean() != 0 else np.nan
            )

            if len(valid_values) >= 2:
                valid_indices = np.where(~pd.isna(values))[0]
                if len(valid_indices) >= 2:
                    slope, _, _, _, _ = stats.linregress(valid_indices, valid_values)
                    features[f"{col}_slope"] = slope
                else:
                    features[f"{col}_slope"] = np.nan
                first_val, last_val = valid_values[0], valid_values[-1]
                features[f"{col}_diff"] = last_val - first_val
                features[f"{col}_ratio"] = last_val / first_val if first_val != 0 else np.nan
            else:
                features[f"{col}_slope"] = features[f"{col}_diff"] = features[f"{col}_ratio"] = np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_monthly_categorical_change_features(monthly_df, employee_ids):
    categorical_cols = ["部署ID", "職種", "役割", "等級", "勤務地", "上司ID"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in categorical_cols:
            if col not in emp_data.columns:
                continue
            values = emp_data[col].values
            changes = sum(1 for i in range(1, len(values)) if pd.notna(values[i]) and pd.notna(values[i-1]) and values[i] != values[i-1])
            features[f"{col}_changes"] = changes
            features[f"{col}_unique_count"] = len(pd.Series(values).dropna().unique())
        if "月末在籍状態" in emp_data.columns:
            status_values = emp_data["月末在籍状態"].values
            features["leave_of_absence_flag"] = int("休職" in status_values)
            features["leave_of_absence_months"] = np.sum(status_values == "休職")
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_missing_value_features(monthly_df, employee_ids):
    missing_target_cols = ["360度評価_親和度", "360度評価_信頼度", "顧客満足度評価", "担当プロジェクト数"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in missing_target_cols:
            if col in emp_data.columns:
                values = emp_data[col].values
                total_months = len(values)
                features[f"{col}_missing_rate"] = pd.isna(values).sum() / total_months if total_months > 0 else np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_domain_knowledge_features(monthly_df, employee_ids):
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        eval_cols = ["360度評価_親和度", "360度評価_信頼度", "360度評価_主体度", "360度評価_学習度", "360度評価_共有貢献度"]
        eval_mean_list = [emp_data[col].mean() for col in eval_cols if col in emp_data.columns]
        features["engagement_score"] = np.nanmean(eval_mean_list) if len(eval_mean_list) > 0 else np.nan
        if "残業時間" in emp_data.columns:
            features["overtime_stability"] = emp_data["残業時間"].std()
        if "研修時間" in emp_data.columns and "残業時間" in emp_data.columns:
            training_mean = emp_data["研修時間"].mean()
            overtime_mean = emp_data["残業時間"].mean()
            features["training_overtime_ratio"] = training_mean / overtime_mean if overtime_mean > 0 else np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_advanced_statistical_features(monthly_df, employee_ids):
    """統計的特徴量：歪度、尖度、パーセンタイル"""
    numeric_cols = ["残業時間", "有給取得日数", "研修時間", "360度評価_親和度", "360度評価_信頼度"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in numeric_cols:
            if col in emp_data.columns:
                values = emp_data[col].dropna().values
                if len(values) >= 3:
                    features[f"{col}_skew"] = stats.skew(values)
                    features[f"{col}_kurtosis"] = stats.kurtosis(values)
                    features[f"{col}_q25"] = np.percentile(values, 25)
                    features[f"{col}_q75"] = np.percentile(values, 75)
                    features[f"{col}_iqr"] = features[f"{col}_q75"] - features[f"{col}_q25"]
                else:
                    features[f"{col}_skew"] = features[f"{col}_kurtosis"] = np.nan
                    features[f"{col}_q25"] = features[f"{col}_q75"] = features[f"{col}_iqr"] = np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_cluster_features(monthly_df, employee_ids, n_clusters=5, seed=42):
    """クラスター特徴量：月次データの平均をKMeansクラスタリング"""
    key_cols = ["残業時間", "有給取得日数", "研修時間", "360度評価_親和度", "360度評価_信頼度"]
    agg_data = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id]
        row = {"社員ID": employee_id}
        for col in key_cols:
            if col in emp_data.columns:
                row[col] = emp_data[col].mean()
        agg_data.append(row)
    agg_df = pd.DataFrame(agg_data)
    feature_cols = [c for c in key_cols if c in agg_df.columns]
    X = agg_df[feature_cols].fillna(-999)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    kmeans = KMeans(n_clusters=n_clusters, random_state=seed, n_init=10)
    agg_df["cluster"] = kmeans.fit_predict(X_scaled)
    return agg_df[["社員ID", "cluster"]]


def create_eda_driven_features(monthly_df, employee_ids):
    """欠勤日数パターン・360度評価タイミング・月次ボラティリティ・比率特徴量（12_〜18_と同一ロジック）"""
    eval_cols = ["360度評価_親和度", "360度評価_信頼度", "360度評価_主体度", "360度評価_学習度", "360度評価_共有貢献度"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}

        absence_vals = emp_data["欠勤日数"].values
        nonzero = absence_vals > 0
        features["欠勤発生月数"] = int(nonzero.sum())
        max_run = cur_run = 0
        for v in nonzero:
            cur_run = cur_run + 1 if v else 0
            max_run = max(max_run, cur_run)
        features["欠勤_最長連続月数"] = max_run
        features["欠勤_連続フラグ"] = int(max_run >= 2)

        flagged = emp_data[emp_data["360度評価更新フラグ"] == 1]
        first_month = flagged["経過月数"].min() if len(flagged) > 0 else np.nan
        features["初回評価月"] = first_month
        features["is_早期評価"] = int(first_month <= 4) if pd.notna(first_month) else 0
        features["is_遅延評価"] = int(first_month >= 7) if pd.notna(first_month) else 0
        features["評価遅延度"] = abs(first_month - 5) if pd.notna(first_month) else np.nan

        for col in ["残業時間", "月例給与_円"]:
            vals = emp_data[col].dropna().values
            features[f"{col}_volatility"] = np.mean(np.abs(np.diff(vals))) if len(vals) >= 2 else np.nan

        n_months = len(emp_data)
        features["有給取得率"] = emp_data["有給取得日数"].sum() / n_months if n_months > 0 else np.nan
        features["評価項目間ばらつき"] = emp_data[eval_cols].std(axis=1).mean()

        features_list.append(features)
    return pd.DataFrame(features_list)


def create_manager_team_size_features(monthly_df, employee_ids):
    """初期（経過月数=0）時点で同じ上司IDを持つ社員数（12_〜18_と同一ロジック）"""
    month0 = monthly_df[monthly_df["経過月数"] == 0].copy()
    month0["初期上司_部下数"] = month0.groupby("上司ID")["社員ID"].transform("count")
    out = month0[["社員ID", "初期上司_部下数"]]
    return out[out["社員ID"].isin(employee_ids)].reset_index(drop=True)

print("✅ split非依存の基本特徴量関数定義完了")


✅ split非依存の基本特徴量関数定義完了


In [9]:
logger.info("-" * 60)
logger.info("split非依存の基本特徴量を生成中...")
logger.info("-" * 60)

train_monthly_agg = create_monthly_aggregation_features(train_monthly, train_ids)
test_monthly_agg = create_monthly_aggregation_features(test_monthly, test_ids)

train_cat_change = create_monthly_categorical_change_features(train_monthly, train_ids)
test_cat_change = create_monthly_categorical_change_features(test_monthly, test_ids)

train_missing = create_missing_value_features(train_monthly, train_ids)
test_missing = create_missing_value_features(test_monthly, test_ids)

train_domain = create_domain_knowledge_features(train_monthly, train_ids)
test_domain = create_domain_knowledge_features(test_monthly, test_ids)

train_advanced_stats = create_advanced_statistical_features(train_monthly, train_ids)
test_advanced_stats = create_advanced_statistical_features(test_monthly, test_ids)

train_cluster = create_cluster_features(train_monthly, train_ids, n_clusters=5, seed=SEED)
test_cluster = create_cluster_features(test_monthly, test_ids, n_clusters=5, seed=SEED)

train_eda_feats = create_eda_driven_features(train_monthly, train_ids)
test_eda_feats = create_eda_driven_features(test_monthly, test_ids)

train_mgr = create_manager_team_size_features(train_monthly, train_ids)
test_mgr = create_manager_team_size_features(test_monthly, test_ids)

logger.info("split非依存の基本特徴量生成完了")


[2026-08-16 13:51:22] [INFO] ------------------------------------------------------------


INFO:71_model_zoo_cpu:------------------------------------------------------------


[2026-08-16 13:51:22] [INFO] split非依存の基本特徴量を生成中...


INFO:71_model_zoo_cpu:split非依存の基本特徴量を生成中...


[2026-08-16 13:51:22] [INFO] ------------------------------------------------------------


INFO:71_model_zoo_cpu:------------------------------------------------------------


[2026-08-16 13:57:09] [INFO] split非依存の基本特徴量生成完了


INFO:71_model_zoo_cpu:split非依存の基本特徴量生成完了


## 2. テキストTF-IDF（A_v1、`51_`と同一・継続採用）

In [10]:
def create_tfidf_svd_features(train_persona, test_persona, col, max_features=300, n_components=15, min_df=3, seed=42):
    '''文字n-gram TF-IDF + TruncatedSVDでテキスト特徴量を生成（Trainのみでfit）'''
    train_text = train_persona[col].fillna("").astype(str)
    test_text = test_persona[col].fillna("").astype(str)

    vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), max_features=max_features, min_df=min_df)
    train_tfidf = vectorizer.fit_transform(train_text)
    test_tfidf = vectorizer.transform(test_text)

    n_comp = min(n_components, train_tfidf.shape[1] - 1)
    svd = TruncatedSVD(n_components=n_comp, random_state=seed, algorithm="arpack")
    train_svd = svd.fit_transform(train_tfidf)
    test_svd = svd.transform(test_tfidf)

    col_names = [f"{col}_tfidf_svd_{i}" for i in range(n_comp)]
    train_out = pd.DataFrame(train_svd, columns=col_names)
    train_out[ID_COL] = train_persona[ID_COL].values
    test_out = pd.DataFrame(test_svd, columns=col_names)
    test_out[ID_COL] = test_persona[ID_COL].values
    return train_out, test_out, svd.explained_variance_ratio_.sum()

TEXT_COLS = ["入社時メモ", "上司からのフィードバック", "同僚からのフィードバック"]

logger.info("テキストTF-IDF+SVD特徴量(A_v1)を生成中...")
tfidf_train_list, tfidf_test_list = [], []
for col in TEXT_COLS:
    tr, te, explained_var = create_tfidf_svd_features(train_persona, test_persona, col, max_features=300, n_components=15, min_df=3, seed=SEED)
    logger.info(f"{col}: SVD累積寄与率={explained_var:.3f}")
    tfidf_train_list.append(tr)
    tfidf_test_list.append(te)

logger.info("テキストTF-IDF+SVD特徴量生成完了")


[2026-08-16 13:57:09] [INFO] テキストTF-IDF+SVD特徴量(A_v1)を生成中...


INFO:71_model_zoo_cpu:テキストTF-IDF+SVD特徴量(A_v1)を生成中...


[2026-08-16 13:57:11] [INFO] 入社時メモ: SVD累積寄与率=0.760


INFO:71_model_zoo_cpu:入社時メモ: SVD累積寄与率=0.760


[2026-08-16 13:57:15] [INFO] 上司からのフィードバック: SVD累積寄与率=0.360


INFO:71_model_zoo_cpu:上司からのフィードバック: SVD累積寄与率=0.360


[2026-08-16 13:57:17] [INFO] 同僚からのフィードバック: SVD累積寄与率=0.421


INFO:71_model_zoo_cpu:同僚からのフィードバック: SVD累積寄与率=0.421


[2026-08-16 13:57:17] [INFO] テキストTF-IDF+SVD特徴量生成完了


INFO:71_model_zoo_cpu:テキストTF-IDF+SVD特徴量生成完了


## 3. 四半期/加速度特徴量（D_expanded、`51_`と同一・継続採用）

In [11]:
def create_quarterly_features(monthly_df, employee_ids, metrics, suffix=""):
    quarters = {"q1": (0, 5), "q2": (6, 11), "q3": (12, 17), "q4": (18, 23)}
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数")
        features = {"社員ID": employee_id}
        for metric in metrics:
            q_means = {}
            for qname, (lo, hi) in quarters.items():
                vals = emp_data[emp_data["経過月数"].between(lo, hi)][metric]
                q_means[qname] = vals.mean()
                features[f"{metric}_{qname}_mean{suffix}"] = q_means[qname]
            first_half_delta = q_means["q2"] - q_means["q1"] if pd.notna(q_means["q1"]) and pd.notna(q_means["q2"]) else np.nan
            second_half_delta = q_means["q4"] - q_means["q3"] if pd.notna(q_means["q3"]) and pd.notna(q_means["q4"]) else np.nan
            features[f"{metric}_acceleration{suffix}"] = (
                second_half_delta - first_half_delta if pd.notna(first_half_delta) and pd.notna(second_half_delta) else np.nan
            )
        features_list.append(features)
    return pd.DataFrame(features_list)

D_EXPANDED_METRICS = [
    "残業時間", "有給取得日数", "欠勤日数", "研修時間",
    "上司との面談実施回数", "情報共有件数", "在宅勤務日数",
    "360度評価_親和度", "360度評価_信頼度", "360度評価_主体度",
    "360度評価_学習度", "360度評価_共有貢献度", "360度評価者数",
    "顧客満足度評価", "担当プロジェクト数", "月例給与_円",
]

logger.info("四半期/加速度特徴量(D_expanded: 16指標)を生成中...")
train_quarterly_exp = create_quarterly_features(train_monthly, train_ids, D_EXPANDED_METRICS, suffix="_exp")
test_quarterly_exp = create_quarterly_features(test_monthly, test_ids, D_EXPANDED_METRICS, suffix="_exp")
logger.info(f"D_expanded: Train {train_quarterly_exp.shape}, Test {test_quarterly_exp.shape}")


[2026-08-16 13:57:17] [INFO] 四半期/加速度特徴量(D_expanded: 16指標)を生成中...


INFO:71_model_zoo_cpu:四半期/加速度特徴量(D_expanded: 16指標)を生成中...


[2026-08-16 13:59:21] [INFO] D_expanded: Train (2761, 81), Test (2502, 81)


INFO:71_model_zoo_cpu:D_expanded: Train (2761, 81), Test (2502, 81)


## 4. Persona単位の基本特徴量（`51_`と同一）

In [12]:
logger.info("Persona単位の基本特徴量を生成中...")
train_persona["入社日"] = pd.to_datetime(train_persona["入社日"])
test_persona["入社日"] = pd.to_datetime(test_persona["入社日"])

for col in TEXT_COLS:
    train_persona[f"{col}_len"] = train_persona[col].fillna("").astype(str).apply(len)
    test_persona[f"{col}_len"] = test_persona[col].fillna("").astype(str).apply(len)
train_persona["text_total_chars"] = train_persona[TEXT_COLS].fillna("").apply(lambda x: sum(len(str(v)) for v in x), axis=1)
test_persona["text_total_chars"] = test_persona[TEXT_COLS].fillna("").apply(lambda x: sum(len(str(v)) for v in x), axis=1)

train_persona["入社年"] = train_persona["入社日"].dt.year
train_persona["入社月"] = train_persona["入社日"].dt.month
train_persona["入社四半期"] = train_persona["入社日"].dt.quarter
test_persona["入社年"] = test_persona["入社日"].dt.year
test_persona["入社月"] = test_persona["入社日"].dt.month
test_persona["入社四半期"] = test_persona["入社日"].dt.quarter

train_persona["年齢_x_前職経験"] = train_persona["入社時年齢"] * train_persona["前職経験月数"]
test_persona["年齢_x_前職経験"] = test_persona["入社時年齢"] * test_persona["前職経験月数"]
grade_map = {"G1": 1, "G2": 2, "G3": 3, "G4": 4, "G5": 5}
train_persona["初期等級_num"] = train_persona["初期等級"].map(grade_map)
test_persona["初期等級_num"] = test_persona["初期等級"].map(grade_map)
train_persona["初任給_x_等級"] = train_persona["初任給_円"] * train_persona["初期等級_num"]
test_persona["初任給_x_等級"] = test_persona["初任給_円"] * test_persona["初期等級_num"]

train_persona["is_Q2_新卒"] = ((train_persona["入社四半期"] == 2) & (train_persona["入社区分"] == "新卒")).astype(int)
test_persona["is_Q2_新卒"] = ((test_persona["入社四半期"] == 2) & (test_persona["入社区分"] == "新卒")).astype(int)

logger.info("Persona単位の基本特徴量処理完了")


[2026-08-16 13:59:21] [INFO] Persona単位の基本特徴量を生成中...


INFO:71_model_zoo_cpu:Persona単位の基本特徴量を生成中...


[2026-08-16 13:59:21] [INFO] Persona単位の基本特徴量処理完了


INFO:71_model_zoo_cpu:Persona単位の基本特徴量処理完了


## 5. 転居×勤務地マッチの交互作用特徴量（ブロックL、v1=`27_`のPublic確認済み版 / v2=抽出拡張版、`51_`と同一）

`51_`と同じくv1/v2両方を生成するが、実際に特徴量として使うのはv2（`BLOCK={"L2"}`）のみ
（`28_`以降ずっとv2が現在の最良）。


In [13]:
def extract_workstyle_section(text):
    if pd.isna(text):
        return None
    m = re.search(r"勤務地・働き方：(.+?)$", text, re.S)
    if m:
        return m.group(1).strip()
    # 50_: 見出しがない書式B（276件、5.24%）のフォールバック（49_で確認済み・Public -0.0022〜-0.0035）。
    lines = [l for l in text.strip().splitlines() if re.search(r"勤務地|転居|在宅勤務", l)]
    return "".join(lines) if lines else None


NEG_RELOC = re.compile(r"転居を伴う(異動|勤務地変更)[はも]?(許容せず|許容していない|許容しておらず|希望せず|希望しておらず|希望していない)")
POS_RELOC = re.compile(r"転居を伴う(異動|勤務地変更)[はもを]?(許容し?ており|許容)")


def classify_reloc(s):
    if s is None:
        return None
    if NEG_RELOC.search(s):
        return False
    if POS_RELOC.search(s):
        return True
    return None


def extract_desired_location_v1(s):
    '''27_・25_・EDA v3/v4/v5と同一（Public 0.529672で確認済み、カバー率88.6%/train）'''
    if s is None:
        return None
    m = re.search(r"(?:勤務地は|希望勤務地は)(.+?)(?:を希望|。)", s)
    if m:
        return m.group(1)
    m2 = re.search(r"(.+?)を希望勤務地", s)
    return m2.group(1) if m2 else None


def extract_desired_location_v2(s):
    '''v1に「◯◯(勤務|での勤務)?を希望。」パターンを追加した拡張版（カバー率94.1%/train）'''
    if s is None:
        return None
    loc = extract_desired_location_v1(s)
    if loc is None:
        m3 = re.search(r"^([一-龥ぁ-んァ-ンー]+?)(?:での勤務|勤務)?を希望。", s)
        loc = m3.group(1) if m3 else None
    if loc is not None:
        loc = loc.strip("「」")
    return loc


def create_relocation_mismatch_features(persona_df, extract_fn, state_col, flag_col):
    ws_section = persona_df["入社時メモ"].apply(extract_workstyle_section)
    reloc_ok_raw = ws_section.apply(classify_reloc)
    desired = ws_section.apply(extract_fn)
    actual = persona_df["初期勤務地"]
    match = (desired == actual) & desired.notna()

    reloc_true = reloc_ok_raw == True
    reloc_false = reloc_ok_raw == False
    valid = desired.notna() & reloc_ok_raw.notna()

    state = pd.Series("unknown", index=persona_df.index)
    state[valid & reloc_true & match] = "許容_一致"
    state[valid & reloc_true & ~match] = "許容_不一致"
    state[valid & reloc_false & match] = "非許容_一致"
    state[valid & reloc_false & ~match] = "非許容_不一致"

    double_bad = (valid & reloc_false & ~match).astype(int)

    return pd.DataFrame({
        "社員ID": persona_df["社員ID"].values,
        state_col: state.values,
        flag_col: double_bad.values,
    })

logger.info("転居×勤務地マッチ交互作用特徴量(ブロックL v1/v2)を生成中...")
train_reloc_v1 = create_relocation_mismatch_features(train_persona, extract_desired_location_v1, "転居x勤務地_状態_v1", "転居x勤務地_ダブル悪条件_v1")
test_reloc_v1 = create_relocation_mismatch_features(test_persona, extract_desired_location_v1, "転居x勤務地_状態_v1", "転居x勤務地_ダブル悪条件_v1")
train_reloc_v2 = create_relocation_mismatch_features(train_persona, extract_desired_location_v2, "転居x勤務地_状態_v2", "転居x勤務地_ダブル悪条件_v2")
test_reloc_v2 = create_relocation_mismatch_features(test_persona, extract_desired_location_v2, "転居x勤務地_状態_v2", "転居x勤務地_ダブル悪条件_v2")

logger.info(f"L_v1: Train {train_reloc_v1.shape}, Test {test_reloc_v1.shape}")
logger.info(f"L_v2: Train {train_reloc_v2.shape}, Test {test_reloc_v2.shape}")


[2026-08-16 13:59:21] [INFO] 転居×勤務地マッチ交互作用特徴量(ブロックL v1/v2)を生成中...


INFO:71_model_zoo_cpu:転居×勤務地マッチ交互作用特徴量(ブロックL v1/v2)を生成中...


[2026-08-16 13:59:21] [INFO] L_v1: Train (2761, 3), Test (2502, 3)


INFO:71_model_zoo_cpu:L_v1: Train (2761, 3), Test (2502, 3)


[2026-08-16 13:59:21] [INFO] L_v2: Train (2761, 3), Test (2502, 3)


INFO:71_model_zoo_cpu:L_v2: Train (2761, 3), Test (2502, 3)


## 6. 部署Target Encoding（リーク対策済）と `prepare_split` 関数（`51_`と同一）

In [14]:
def create_department_target_encoding(train_persona, test_persona, y_train, fit_ids, seed=42, n_splits=5, smoothing=10):
    '''初期部署IDのKFold + スムージング付きTarget Encoding（15_〜18_の修正版と同一ロジック）'''
    col = "初期部署ID"
    is_fit = train_persona[ID_COL].isin(fit_ids).values
    dept_all = train_persona[col].values
    y_arr = y_train.values
    global_mean = y_arr[is_fit].mean()

    fit_indices = np.where(is_fit)[0]
    dept_fit = dept_all[fit_indices]
    y_fit = y_arr[fit_indices]

    train_te = np.full(len(train_persona), global_mean)

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for tr_idx, val_idx in kf.split(fit_indices):
        df_tr = pd.DataFrame({col: dept_fit[tr_idx], "y": y_fit[tr_idx]})
        stats_tr = df_tr.groupby(col)["y"].agg(["mean", "count"])
        smoothed = (stats_tr["count"] * stats_tr["mean"] + smoothing * global_mean) / (stats_tr["count"] + smoothing)
        mapping = smoothed.to_dict()
        actual_val_idx = fit_indices[val_idx]
        train_te[actual_val_idx] = pd.Series(dept_fit[val_idx]).map(mapping).fillna(global_mean).values

    df_full = pd.DataFrame({col: dept_fit, "y": y_fit})
    stats_full = df_full.groupby(col)["y"].agg(["mean", "count"])
    smoothed_full = (stats_full["count"] * stats_full["mean"] + smoothing * global_mean) / (stats_full["count"] + smoothing)
    mapping_full = smoothed_full.to_dict()
    dept_size_map = stats_full["count"].to_dict()

    not_fit_indices = np.where(~is_fit)[0]
    train_te[not_fit_indices] = pd.Series(dept_all[not_fit_indices]).map(mapping_full).fillna(global_mean).values

    test_te = test_persona[col].map(mapping_full).fillna(global_mean).values

    train_out = pd.DataFrame({
        ID_COL: train_persona[ID_COL].values,
        "dept_target_enc": train_te,
        "dept_size": pd.Series(dept_all).map(dept_size_map).fillna(0).values,
    })
    test_out = pd.DataFrame({
        ID_COL: test_persona[ID_COL].values,
        "dept_target_enc": test_te,
        "dept_size": test_persona[col].map(dept_size_map).fillna(0).values,
    })
    return train_out, test_out


def prepare_split(split_ratio, extra_blocks=None, exclude_early_from_val=True):
    '''指定した分割比率で特徴量を組み立てる（51_と完全に同一ロジック）。'''
    extra_blocks = extra_blocks or set()
    sorted_persona = train_persona.sort_values("入社日")
    split_point = int(len(sorted_persona) * split_ratio)
    train_period_ids = set(sorted_persona.iloc[:split_point][ID_COL])

    train_dept_te, test_dept_te = create_department_target_encoding(
        train_persona, test_persona, y_train, fit_ids=train_period_ids, seed=SEED, n_splits=5, smoothing=10
    )

    train_persona_features = train_persona.drop(columns=[TARGET_COL])
    tf = train_persona_features.merge(train_monthly_agg, on=ID_COL, how="left")
    tf = tf.merge(train_cat_change, on=ID_COL, how="left")
    tf = tf.merge(train_missing, on=ID_COL, how="left")
    tf = tf.merge(train_domain, on=ID_COL, how="left")
    tf = tf.merge(train_advanced_stats, on=ID_COL, how="left")
    tf = tf.merge(train_cluster, on=ID_COL, how="left")
    tf = tf.merge(train_dept_te, on=ID_COL, how="left")
    tf = tf.merge(train_eda_feats, on=ID_COL, how="left")
    tf = tf.merge(train_mgr, on=ID_COL, how="left")
    tf = tf.merge(train_quarterly_exp, on=ID_COL, how="left")
    for trdf in tfidf_train_list:
        tf = tf.merge(trdf, on=ID_COL, how="left")

    ttf = test_persona.merge(test_monthly_agg, on=ID_COL, how="left")
    ttf = ttf.merge(test_cat_change, on=ID_COL, how="left")
    ttf = ttf.merge(test_missing, on=ID_COL, how="left")
    ttf = ttf.merge(test_domain, on=ID_COL, how="left")
    ttf = ttf.merge(test_advanced_stats, on=ID_COL, how="left")
    ttf = ttf.merge(test_cluster, on=ID_COL, how="left")
    ttf = ttf.merge(test_dept_te, on=ID_COL, how="left")
    ttf = ttf.merge(test_eda_feats, on=ID_COL, how="left")
    ttf = ttf.merge(test_mgr, on=ID_COL, how="left")
    ttf = ttf.merge(test_quarterly_exp, on=ID_COL, how="left")
    for tedf in tfidf_test_list:
        ttf = ttf.merge(tedf, on=ID_COL, how="left")

    if "L1" in extra_blocks:
        tf = tf.merge(train_reloc_v1, on=ID_COL, how="left")
        ttf = ttf.merge(test_reloc_v1, on=ID_COL, how="left")

    if "L2" in extra_blocks:
        tf = tf.merge(train_reloc_v2, on=ID_COL, how="left")
        ttf = ttf.merge(test_reloc_v2, on=ID_COL, how="left")

    _train_period_features = tf[tf[ID_COL].isin(train_period_ids)]
    job_dev_metrics = ["残業時間_mean", "研修時間_mean", "360度評価_親和度_mean"]
    job_means = {m: _train_period_features.groupby("初期職種")[m].mean().to_dict() for m in job_dev_metrics}
    category_means_train = {m: _train_period_features.groupby("入社区分")[m].mean().to_dict() for m in job_dev_metrics}
    grade_salary_mean = _train_period_features.groupby("初期等級")["初任給_円"].mean().to_dict()
    category_salary_mean = _train_period_features.groupby("入社区分")["初任給_円"].mean().to_dict()
    grade_monthly_salary_mean = _train_period_features.groupby("初期等級")["月例給与_円_mean"].mean().to_dict()

    for df_ in [tf, ttf]:
        for m in job_dev_metrics:
            df_[f"{m}_job_deviation"] = df_[m] - df_["初期職種"].map(job_means[m])
        df_["研修時間_職種比"] = df_["研修時間_mean"] / df_["初期職種"].map(job_means["研修時間_mean"]).replace(0, np.nan)
        df_["研修時間_区分比"] = df_["研修時間_mean"] / df_["入社区分"].map(category_means_train["研修時間_mean"]).replace(0, np.nan)
        df_["初任給_等級内偏差"] = df_["初任給_円"] - df_["初期等級"].map(grade_salary_mean)
        df_["初任給_区分内偏差"] = df_["初任給_円"] - df_["入社区分"].map(category_salary_mean)
        df_["月例給与_等級内偏差"] = df_["月例給与_円_mean"] - df_["初期等級"].map(grade_monthly_salary_mean)

    drop_cols = ["入社時メモ", "上司からのフィードバック", "同僚からのフィードバック",
                 "初期部署ID", "初期等級", "最終学歴", "前職職種"]
    tf = tf.drop(columns=[c for c in drop_cols if c in tf.columns]).set_index(ID_COL)
    ttf = ttf.drop(columns=[c for c in drop_cols if c in ttf.columns]).set_index(ID_COL)

    target_series = train_persona.set_index(ID_COL)[TARGET_COL]
    tf_sorted = tf.sort_values("入社日")
    y_sorted = target_series.loc[tf_sorted.index]

    ag_train = tf_sorted.iloc[:split_point].copy()
    ag_tuning = tf_sorted.iloc[split_point:].copy()
    ag_train[TARGET_COL] = y_sorted.iloc[:split_point].values
    ag_tuning[TARGET_COL] = y_sorted.iloc[split_point:].values

    if exclude_early_from_val and len(ag_tuning) > 0:
        n_before = len(ag_tuning)
        ag_tuning = ag_tuning[~ag_tuning.index.isin(EARLY_LEAVER_IDS)]
        logger.info(f"  検証セット: {n_before} → {len(ag_tuning)}件（早期退職者{n_before - len(ag_tuning)}名を除外）")

    return ag_train, ag_tuning, ttf

print("✅ 部署Target Encoding・prepare_split関数定義完了")


✅ 部署Target Encoding・prepare_split関数定義完了


## 7. 特徴量の組み立て（`51_`と同一、`BLOCK={"L2"}`固定）

In [15]:
def _feature_cols(df):
    return [c for c in df.columns if c not in ["入社日", TARGET_COL]]


BLOCK = {"L2"}   # 28_のL_v2_extended（現在の最良）に固定

logger.info("=" * 60)
logger.info("[検証用] split_80_20 / 検証=生存者のみ")
ag_train_80b, ag_val_surv, _ = prepare_split(0.8, extra_blocks=BLOCK, exclude_early_from_val=True)

logger.info("[提出用] 全件学習（検証セットなし）")
ag_full, ag_empty, test_features_full = prepare_split(1.0, extra_blocks=BLOCK, exclude_early_from_val=True)

logger.info("-" * 60)
logger.info(f"main_train={len(ag_train_80b)}, main_valid(生存者)={len(ag_val_surv)}")
logger.info(f"全件={len(ag_full)}")
logger.info(f"特徴量数: {len(_feature_cols(ag_train_80b))}")
assert len(ag_empty) == 0, "全件学習のときは検証セットが空のはず"


[2026-08-16 13:59:21] [INFO] ============================================================


INFO:71_model_zoo_cpu:============================================================


[2026-08-16 13:59:21] [INFO] [検証用] split_80_20 / 検証=生存者のみ


INFO:71_model_zoo_cpu:[検証用] split_80_20 / 検証=生存者のみ


[2026-08-16 13:59:22] [INFO]   検証セット: 553 → 535件（早期退職者18名を除外）


...

INFO:71_model_zoo_cpu:main_train=2208, main_valid(生存者)=535


[2026-08-16 13:59:22] [INFO] 全件=2761


INFO:71_model_zoo_cpu:全件=2761


[2026-08-16 13:59:22] [INFO] 特徴量数: 441


INFO:71_model_zoo_cpu:特徴量数: 441


## 8. 特徴量グループの棚卸し（`51_`から移植、内容は同一）

In [16]:
ALL_FEATS = set(_feature_cols(ag_train_80b))

DERIVED_COLS = [
    "残業時間_mean_job_deviation", "研修時間_mean_job_deviation", "360度評価_親和度_mean_job_deviation",
    "研修時間_職種比", "研修時間_区分比",
    "初任給_等級内偏差", "初任給_区分内偏差", "月例給与_等級内偏差",
]
DEPT_TE_COLS = ["dept_target_enc", "dept_size"]


def _cols_of(df):
    return [c for c in df.columns if c != ID_COL]


_RAW_GROUPS = {
    "persona":   [c for c in train_persona.columns if c not in (ID_COL, TARGET_COL)],
    "agg":       _cols_of(train_monthly_agg),
    "catchange": _cols_of(train_cat_change),
    "missing":   _cols_of(train_missing),
    "domain":    _cols_of(train_domain),
    "advstats":  _cols_of(train_advanced_stats),
    "cluster":   _cols_of(train_cluster),
    "deptte":    DEPT_TE_COLS,
    "edafeat":   _cols_of(train_eda_feats),
    "mgr":       _cols_of(train_mgr),
    "quarterly": _cols_of(train_quarterly_exp),
    "tfidf":     [c for _df in tfidf_train_list for c in _cols_of(_df)],
    "L2":        _cols_of(train_reloc_v2),
    "derived":   DERIVED_COLS,
}

FEATURE_GROUPS = {g: [c for c in cols if c in ALL_FEATS] for g, cols in _RAW_GROUPS.items()}
ALL_GROUPS = set(FEATURE_GROUPS)

_covered = [c for cols in FEATURE_GROUPS.values() for c in cols]
_dupes = sorted({c for c in _covered if _covered.count(c) > 1})
assert not _dupes, f"複数グループに重複している列: {_dupes}"
_orphans = sorted(ALL_FEATS - set(_covered))
assert not _orphans, f"どのグループにも属さない列: {_orphans}"

print(f"特徴量 合計 {len(ALL_FEATS)} 列")
print("-" * 52)
for g in sorted(FEATURE_GROUPS, key=lambda x: -len(FEATURE_GROUPS[x])):
    print(f"  {g:<10s} {len(FEATURE_GROUPS[g]):>4d} 列   例: {FEATURE_GROUPS[g][:2]}")
print("-" * 52)
print("✅ グループ分類は全列を過不足なく覆っている")


特徴量 合計 441 列
----------------------------------------------------
  agg         224 列   例: ['残業時間_mean', '残業時間_std']
  quarterly    80 列   例: ['残業時間_q1_mean_exp', '残業時間_q2_mean_exp']
  tfidf        45 列   例: ['入社時メモ_tfidf_svd_0', '入社時メモ_tfidf_svd_1']
  advstats     25 列   例: ['残業時間_skew', '残業時間_kurtosis']
  persona      21 列   例: ['入社区分', '入社時年齢']
  catchange    14 列   例: ['部署ID_changes', '部署ID_unique_count']
  edafeat      11 列   例: ['欠勤発生月数', '欠勤_最長連続月数']
  derived       8 列   例: ['残業時間_mean_job_deviation', '研修時間_mean_job_deviation']
  missing       4 列   例: ['360度評価_親和度_missing_rate', '360度評価_信頼度_missing_rate']
  domain        3 列   例: ['engagement_score', 'overtime_stability']
  deptte        2 列   例: ['dept_target_enc', 'dept_size']
  L2            2 列   例: ['転居x勤務地_状態_v2', '転居x勤務地_ダブル悪条件_v2']
  cluster       1 列   例: ['cluster']
  mgr           1 列   例: ['初期上司_部下数']
----------------------------------------------------
✅ グループ分類は全列を過不足なく覆っている


---
## 9. 設定（事前登録した内容をコードに落とす）

In [17]:
import time, gc
from sklearn.metrics import log_loss
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

# ---- CatBoost基準（54_/68_ と同一設定。ここは探索しない） ----
A_PARAMS = {"depth": 4, "learning_rate": 0.03518359458951149, "l2_leaf_reg": 2.217690447016724,
            "border_count": 218, "bagging_temperature": 0.6787467566574921,
            "random_strength": 1.438494697238285}
ITER_FIXED = 560
SEEDS_CB = [42, 2024, 7, 1234, 99]

# ---- 事前登録したゲート ----
GATE_VAL_MARGIN = 0.02    # (A) 足切り: CatBoost val + これ以内
GATE_CORR_MAX   = 0.96    # (B) 多様性: CatBoost val予測との相関がこれ未満
NOISE_HINT      = 0.003   # 単一シードsdの目安。これ以下の差は読まない

# ---- 実行制御 ----
N_TRIALS      = 12        # Optuna試行数（内側CVのみ。データが小さいので少なくてよい）
INNER_FOLDS   = 3         # Optunaの内側CV
SEEDS_ZOO     = [42, 2024, 7]
RECOMPUTE_ALL = False     # Trueで既存キャッシュを無視して全部計算し直す
SKIP_MODELS   = []        # 例: ["gp"] でGPを飛ばす（GPは n=2208 で数分かかる）

FEATS_ALL = _feature_cols(ag_train_80b)
y_tr  = ag_train_80b[TARGET_COL].values
y_val = ag_val_surv[TARGET_COL].values
assert len(FEATS_ALL) == 441, f"{len(FEATS_ALL)}列（441列のはず）"
assert len(ag_val_surv) == 535, f"検証{len(ag_val_surv)}名（535名のはず。68_の保存予測と揃わない）"
print(f"全特徴量 {len(FEATS_ALL)} 列 / 学習 {len(ag_train_80b)}件 / 検証 {len(ag_val_surv)}件 / 全件 {len(ag_full)}件")
print(f"ゲート: val ≤ CatBoost+{GATE_VAL_MARGIN} かつ corr < {GATE_CORR_MAX}")


全特徴量 441 列 / 学習 2208件 / 検証 535件 / 全件 2761件
ゲート: val ≤ CatBoost+0.02 かつ corr < 0.96


## 10. CatBoost 基準（ビュー `full441`、重要度の供給源も兼ねる）

`68_` cell 29 と同一。ここは**探索しない**（[[hyperparameter-retuning-exhausted]]）。

In [18]:
def cb_run(feats, train_df, pred_df, seeds=SEEDS_CB, iters=ITER_FIXED):
    obj = [c for c in feats if train_df[c].dtype == "object"]
    Xtr, ytr = train_df[feats].fillna(-999), train_df[TARGET_COL]
    Xpr = pred_df[feats].fillna(-999)
    ps, ms = [], []
    for s in seeds:
        m = cb.CatBoostClassifier(**A_PARAMS, iterations=int(iters), random_seed=s, verbose=False,
                                  cat_features=obj, task_type="CPU")
        m.fit(Xtr, ytr); ms.append(m); ps.append(m.predict_proba(Xpr)[:, 1])
    return np.mean(ps, axis=0), ms


t0 = time.time()
cb_val, cb_models = cb_run(FEATS_ALL, ag_train_80b, ag_val_surv)
CB_VAL = log_loss(y_val, cb_val)
print(f"CatBoost(full441) val = {CB_VAL:.6f}  ({time.time()-t0:.0f}秒)")

imp = pd.Series(np.mean([m.get_feature_importance() for m in cb_models], axis=0),
                index=FEATS_ALL).sort_values(ascending=False)
imp.to_csv(OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_catboost_importance.csv")

# 全件学習でのTest予測（反復数は件数比1.25でスケール。37_で確定した運用）
t0 = time.time()
cb_test, _ = cb_run(FEATS_ALL, ag_full, test_features_full, iters=ITER_FIXED * 1.25)
print(f"CatBoost Test予測 完了 ({time.time()-t0:.0f}秒)  平均 {cb_test.mean():.4f}")


CatBoost(full441) val = 0.516147  (25秒)
CatBoost Test予測 完了 (34秒)  平均 0.5898


## 11. ビューの構築

`68_` cell 31 と同じ作り方（`cb_topK` / `hire_fixed` / `full441`）に、
カーネル法用の `num_top50_std`（**数値列だけ**を重要度上位50本、標準化）を追加する。
すべて**学習データのみ**から作る。

In [19]:
HIRE_GROUPS = ["persona", "deptte", "derived", "L2", "tfidf"]
hire_cols = {c for g in HIRE_GROUPS if g in FEATURE_GROUPS for c in FEATURE_GROUPS[g]}

_obj_cols = {c for c in FEATS_ALL if ag_train_80b[c].dtype == "object"}
_num_rank = [c for c in imp.index if c not in _obj_cols]

VIEWS = {
    "full441":       FEATS_ALL,
    "cb_top150":     imp.head(150).index.tolist(),
    "hire_fixed":    [c for c in FEATS_ALL if c in hire_cols],
    "num_top50_std": _num_rank[:50],
}
for k, v in VIEWS.items():
    print(f"  {k:14s} {len(v):4d}列  （うちカテゴリ {sum(1 for c in v if c in _obj_cols)}本）")

# ---- 事前登録した割り当て（結果を見てから変えない） ----
ASSIGN = {
    "ebm":        "cb_top150",
    "gp":         "num_top50_std",
    "svm_rbf":    "num_top50_std",
    "mlp":        "hire_fixed",
    "extratrees": "full441",
    "logreg":     "hire_fixed",
}
print("\n事前登録した モデル→ビュー の割り当て:")
for m, v in ASSIGN.items():
    print(f"  {m:12s} → {v:14s} ({len(VIEWS[v])}列)")


  full441         441列  （うちカテゴリ 8本）
  cb_top150       150列  （うちカテゴリ 4本）
  hire_fixed       78列  （うちカテゴリ 8本）
  num_top50_std    50列  （うちカテゴリ 0本）

事前登録した モデル→ビュー の割り当て:
  ebm          → cb_top150      (150列)
  gp           → num_top50_std  (50列)
  svm_rbf      → num_top50_std  (50列)
  mlp          → hire_fixed     (78列)
  extratrees   → full441        (441列)
  logreg       → hire_fixed     (78列)


## 12. 行列の作り方（モデルの機構ごとに変える）

- **木系（ExtraTrees）**: カテゴリは序数コード、欠損は -999
- **カーネル/線形/NN（GP・SVM・MLP・Logistic）**: カテゴリは One-Hot、数値は中央値補完 → 標準化
  （`fit` は**学習データのみ**、検証・Testは `transform` のみ）
- **EBM**: 生の DataFrame をそのまま渡す（欠損もカテゴリもネイティブに扱える）

In [20]:
def make_ordinal(feats, fit_df, dfs):
    '''木系用。カテゴリは学習データ基準の序数コード、欠損は -999。'''
    cats = {c: pd.Index(fit_df[c].astype(str).unique()) for c in feats if fit_df[c].dtype == "object"}
    out = []
    for d in dfs:
        M = d[feats].copy()
        for c, idx in cats.items():
            M[c] = idx.get_indexer(M[c].astype(str)).astype(float)   # 未知値は -1
        out.append(M.astype(np.float64).replace([np.inf, -np.inf], np.nan).fillna(-999).values)
    return out


def make_dense_std(feats, fit_df, dfs):
    '''カーネル/線形/NN用。One-Hot + 中央値補完 + 標準化。fitは学習データのみ。'''
    obj = [c for c in feats if fit_df[c].dtype == "object"]
    num = [c for c in feats if c not in obj]
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False, min_frequency=10)
    ohe.fit(fit_df[obj].astype(str)) if obj else None
    imp_ = SimpleImputer(strategy="median")
    sc = StandardScaler()
    Nf = fit_df[num].astype(np.float64).replace([np.inf, -np.inf], np.nan)
    # 全行NaNの列は中央値もNaNになり SimpleImputer が列ごと落とすため、先に0で潰す
    Nf = Nf.fillna(Nf.median()).fillna(0.0)
    sc.fit(imp_.fit_transform(Nf))
    out = []
    for d in dfs:
        N = d[num].astype(np.float64).replace([np.inf, -np.inf], np.nan)
        N = N.fillna(Nf.median()).fillna(0.0)
        X = sc.transform(imp_.transform(N))
        if obj:
            X = np.hstack([X, ohe.transform(d[obj].astype(str))])
        out.append(np.ascontiguousarray(X, dtype=np.float64))
    return out


def make_raw(feats, fit_df, dfs):
    '''EBM用。生のDataFrameをそのまま渡す。'''
    return [d[feats].copy() for d in dfs]


## 13. モデル定義（Optunaの探索空間は内側CVでのみ使う）

`GaussianProcessClassifier` はカーネルのハイパーパラメータを**周辺尤度で内部最適化**するため
Optuna は回さない（回すと二重最適化になる）。

In [21]:
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.gaussian_process.kernels import RBF, ConstantKernel, WhiteKernel
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
try:
    from interpret.glassbox import ExplainableBoostingClassifier
    HAS_EBM = True
except Exception as e:
    HAS_EBM = False
    print(f"⚠️ interpret を読み込めなかったので EBM は飛ばす: {e}")


def _space(kind, t):
    if kind == "ebm":
        return dict(max_bins=t.suggest_categorical("max_bins", [128, 256]),
                    interactions=t.suggest_int("interactions", 0, 15),
                    learning_rate=t.suggest_float("learning_rate", 0.005, 0.05, log=True),
                    min_samples_leaf=t.suggest_int("min_samples_leaf", 2, 20))
    if kind == "svm_rbf":
        return dict(C=t.suggest_float("C", 1e-2, 1e2, log=True),
                    gamma=t.suggest_float("gamma", 1e-4, 1e-1, log=True))
    if kind == "mlp":
        return dict(hidden_layer_sizes=t.suggest_categorical("hidden", [(64,), (128,), (64, 32), (128, 64)]),
                    alpha=t.suggest_float("alpha", 1e-5, 1e0, log=True),
                    learning_rate_init=t.suggest_float("lr", 1e-4, 1e-2, log=True))
    if kind == "extratrees":
        return dict(n_estimators=600,
                    max_depth=t.suggest_int("max_depth", 4, 20),
                    min_samples_leaf=t.suggest_int("min_samples_leaf", 1, 30),
                    max_features=t.suggest_float("max_features", 0.1, 0.9))
    if kind == "logreg":
        return dict(C=t.suggest_float("C", 1e-4, 1e2, log=True))
    return {}


def _build(kind, params, seed):
    if kind == "ebm":
        return ExplainableBoostingClassifier(random_state=seed, n_jobs=-1, **params)
    if kind == "gp":
        k = ConstantKernel(1.0) * RBF(length_scale=10.0) + WhiteKernel(1e-2)
        return GaussianProcessClassifier(kernel=k, random_state=seed, n_restarts_optimizer=0)
    if kind == "svm_rbf":
        return SVC(probability=True, random_state=seed, **params)
    if kind == "mlp":
        return MLPClassifier(max_iter=600, early_stopping=True, n_iter_no_change=20,
                             random_state=seed, **params)
    if kind == "extratrees":
        return ExtraTreesClassifier(random_state=seed, n_jobs=-1, **params)
    if kind == "logreg":
        return LogisticRegression(max_iter=3000, random_state=seed, **params)
    raise ValueError(kind)


PREP = {"ebm": make_raw, "gp": make_dense_std, "svm_rbf": make_dense_std,
        "mlp": make_dense_std, "extratrees": make_ordinal, "logreg": make_dense_std}
TUNABLE = {"ebm", "svm_rbf", "mlp", "extratrees", "logreg"}          # gp は内部最適化するので除く
STOCHASTIC = {"ebm", "mlp", "extratrees"}                            # 複数シードで平均する対象


## 14. 実行

各モデルについて:
1. **`ag_train_80b` の内側3-fold CV** で Optuna（検証535名は一切見ない）
2. 確定パラメータで `ag_train_80b` 全体を学習 → **検証535名の予測**
3. 同じパラメータで `ag_full`(2761名) を学習 → **Test 2502名の予測**

結果は `.npy` にキャッシュする。2回目以降は**キャッシュを読むだけ**になり、その旨をログに出す。

In [22]:
def _cache(name, tag):
    return OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_{name}_{tag}.npy"


def run_model(kind):
    view = ASSIGN[kind]; feats = VIEWS[view]
    pv, pt = _cache(kind, "valpreds"), _cache(kind, "testpreds")
    if not RECOMPUTE_ALL and pv.exists() and pt.exists():
        v, t = np.load(pv), np.load(pt)
        msg = f"【再利用】{kind}: 本NBの過去実行のキャッシュを読み込んだ（再学習していない） {pv.name}"
        print(msg); logger.info(msg)
        # キャッシュはシード平均後の1本しか持たないので、シード間sdは「不明」= NaN。
        # ここを 0.0 にすると「シード分散ゼロ」と誤読される。
        return v, t, None, float("nan")

    prep = PREP[kind]
    Xtr, Xva = prep(feats, ag_train_80b, [ag_train_80b, ag_val_surv])
    Xfu, Xte = prep(feats, ag_full, [ag_full, test_features_full])
    yfu = ag_full[TARGET_COL].values

    best = {}
    if kind in TUNABLE:
        def obj(t):
            p = _space(kind, t); sc = []
            for tri, vai in StratifiedKFold(INNER_FOLDS, shuffle=True, random_state=0).split(Xtr, y_tr):
                m = _build(kind, p, 42)
                m.fit(Xtr[tri] if not isinstance(Xtr, pd.DataFrame) else Xtr.iloc[tri], y_tr[tri])
                q = m.predict_proba(Xtr[vai] if not isinstance(Xtr, pd.DataFrame) else Xtr.iloc[vai])[:, 1]
                sc.append(log_loss(y_tr[vai], q))
            return float(np.mean(sc))
        st = optuna.create_study(direction="minimize",
                                 sampler=optuna.samplers.TPESampler(seed=42))
        st.optimize(obj, n_trials=N_TRIALS, show_progress_bar=False)
        best = _space(kind, optuna.trial.FixedTrial(st.best_params))
        print(f"    内側CV最良 = {st.best_value:.6f}  params={st.best_params}")

    seeds = SEEDS_ZOO if kind in STOCHASTIC else [42]
    vps, tps = [], []
    for s in seeds:
        m = _build(kind, best, s); m.fit(Xtr, y_tr); vps.append(m.predict_proba(Xva)[:, 1])
        m2 = _build(kind, best, s); m2.fit(Xfu, yfu); tps.append(m2.predict_proba(Xte)[:, 1])
        del m, m2; gc.collect()
    v, t = np.mean(vps, 0), np.mean(tps, 0)
    np.save(pv, v); np.save(pt, t)
    return v, t, best, float(np.std([log_loss(y_val, x) for x in vps])) if len(vps) > 1 else 0.0


ZOO = {"catboost": dict(view="full441", val=cb_val, test=cb_test, params=A_PARAMS, sd=np.nan,
                        source="本NBで新規学習")}
for kind in ASSIGN:
    if kind in SKIP_MODELS or (kind == "ebm" and not HAS_EBM):
        print(f"— {kind}: スキップ"); continue
    print(f"\n▶ {kind}  (view={ASSIGN[kind]}, {len(VIEWS[ASSIGN[kind]])}列)")
    t0 = time.time()
    v, t, p, sd = run_model(kind)
    ZOO[kind] = dict(view=ASSIGN[kind], val=v, test=t, params=p, sd=sd,
                     source="キャッシュ再利用" if p is None else "本NBで新規学習")
    print(f"    val = {log_loss(y_val, v):.6f}  (シード間sd {sd:.5f})  {time.time()-t0:.0f}秒")



▶ ebm  (view=cb_top150, 150列)
    内側CV最良 = 0.521740  params={'max_bins': 128, 'interactions': 1, 'learning_rate': 0.024165903162442326, 'min_samples_leaf': 10}
    val = 0.514536  (シード間sd 0.00093)  624秒

▶ gp  (view=num_top50_std, 50列)
    val = 0.571202  (シード間sd 0.00000)  202秒

▶ svm_rbf  (view=num_top50_std, 50列)
    内側CV最良 = 0.560952  params={'C': 75.36364887850348, 'gamma': 0.00010422971466648466}
    val = 0.573573  (シード間sd 0.00000)  18秒

▶ mlp  (view=hire_fixed, 78列)
    内側CV最良 = 0.588101  params={'hidden': (64,), 'alpha': 0.0008161896474305741, 'lr': 0.0016809324835420135}
    val = 0.566216  (シード間sd 0.02052)  19秒

▶ extratrees  (view=full441, 441列)
    内側CV最良 = 0.544194  params={'max_depth': 18, 'min_samples_leaf': 7, 'max_features': 0.24545997376568052}
    val = 0.535172  (シード間sd 0.00086)  93秒

▶ logreg  (view=hire_fixed, 78列)
    内側CV最良 = 0.561718  params={'C': 0.02977751217076364}
    val = 0.551791  (シード間sd 0.00000)  1秒


## 15. 【再利用】過去の実行結果の読み込み

ここから先で使う予測は **本ノートブックでは一切学習していない**。すべて保存済みファイルの読み込みである。
GPU が必要な TabPFN / TabICL / TabDPT はここで読み込むだけで、学習は行わない。

In [23]:
def _find(pattern):
    hits = sorted((PROJECT_ROOT / "data" / "output").glob(pattern))
    return hits[-1] if hits else None


REUSED = {}   # 名前 -> dict(val=..., test=..., src=..., note=...)

# --- (1) 68_ の基盤モデル val予測（535名）。相関の把握にのみ使う ---
for fm in ["tabpfn", "tabicl", "tabdpt"]:
    p = _find(f"*/*_68_foundation_models_3way_{fm}_hire_fixed_valpreds.npy")
    q = _find(f"*/*_70_hire_fixed_fm_submission_{fm}_hire_fixed_testpreds.npy")
    if p is None and q is None:
        print(f"  （{fm}: 保存予測が見つからないので相関比較から除外）"); continue
    v = np.load(p) if p is not None else None
    t = np.load(q) if q is not None else None
    REUSED[f"{fm}(hire_fixed)"] = dict(val=v, test=t)
    msg = (f"【再利用】{fm}(hire_fixed): val={p.name if p else 'なし'} / test={q.name if q else 'なし'}"
           f" — 本NBでは学習していない（GPU必須のため）")
    print(msg); logger.info(msg)
print("  ⚠️ 68_のval予測は速度優先の縮小設定(TabPFN n_estimators=4)。絶対値の比較には使わず相関のみ見る")
print("  ⚠️ 70_のtest予測は本番設定（Public 0.511832 を出した構成の素材）")

# --- (2) AutoGluon プールC（Test予測のみ。valは存在しない） ---
pc = _find("*/*_pool_poolC_weighted.csv")
if pc is not None:
    # ⚠️ prepare_split は set_index(社員ID) しているので、Test の並びは .index で取る
    poolC = pd.read_csv(pc, header=None, names=[ID_COL, "p"]).set_index(ID_COL)
    assert set(poolC.index) == set(test_features_full.index), "プールCとTestの社員IDが一致しない"
    poolC = poolC.loc[test_features_full.index, "p"].values
    REUSED["poolC(AutoGluon8本平均)"] = dict(val=None, test=poolC)
    msg = (f"【再利用】poolC: {pc.name} — 50_/51_/53_/61_×2/62_×4 の full441 weighted 8本平均。"
           f"本NBでは AutoGluon を一切実行していない。64_で追加中のシードは未反映")
    print(msg); logger.info(msg)
else:
    poolC = None
    print("  （プールCのCSVが見つからない。相関比較から除外）")

# --- (3) 54_ の単層CatBoost最良 val予測（参考照合） ---
p54 = _find("*/*_54_l2_m_interaction_R0_memofix_plus_LM_valpreds.npy")
if p54 is not None:
    v54 = np.load(p54); v54 = v54.mean(0) if v54.ndim == 2 else v54
    msg = (f"【再利用】54_ CatBoost最良 val予測: {p54.name}（8シード平均） — "
           f"本NBのCatBoost基準との照合用。相関 {np.corrcoef(v54, cb_val)[0,1]:.4f}, "
           f"val {log_loss(y_val, v54):.6f} vs 本NB {CB_VAL:.6f}")
    print(msg); logger.info(msg)


【再利用】tabpfn(hire_fixed): val=20260816_68_foundation_models_3way_tabpfn_hire_fixed_valpreds.npy / test=20260816_70_hire_fixed_fm_submission_tabpfn_hire_fixed_testpreds.npy — 本NBでは学習していない（GPU必須のため）
[2026-08-16 14:16:20] [INFO] 【再利用】tabpfn(hire_fixed): val=20260816_68_foundation_models_3way_tabpfn_hire_fixed_valpreds.npy / test=20260816_70_hire_fixed_fm_submission_tabpfn_hire_fixed_testpreds.npy — 本NBでは学習していない（GPU必須のため）


INFO:71_model_zoo_cpu:【再利用】tabpfn(hire_fixed): val=20260816_68_foundation_models_3way_tabpfn_hire_fixed_valpreds.npy / test=20260816_70_hire_fixed_fm_submission_tabpfn_hire_fixed_testpreds.npy — 本NBでは学習していない（GPU必須のため）


【再利用】tabicl(hire_fixed): val=20260816_68_foundation_models_3way_tabicl_hire_fixed_valpreds.npy / test=20260816_70_hire_fixed_fm_submission_tabicl_hire_fixed_testpreds.npy — 本NBでは学習していない（GPU必須のため）
[2026-08-16 14:16:21] [INFO] 【再利用】tabicl(hire_fixed): val=20260816_68_foundation_models_3way_tabicl_hire_fixed_valpreds.npy / test=20260816_70_hire_fixed_fm_submission_tabicl_hire_fixed_testpreds.npy — 本NBでは学習していない（GPU必須のため）


INFO:71_model_zoo_cpu:【再利用】tabicl(hire_fixed): val=20260816_68_foundation_models_3way_tabicl_hire_fixed_valpreds.npy / test=20260816_70_hire_fixed_fm_submission_tabicl_hire_fixed_testpreds.npy — 本NBでは学習していない（GPU必須のため）


【再利用】tabdpt(hire_fixed): val=20260816_68_foundation_models_3way_tabdpt_hire_fixed_valpreds.npy / test=20260816_70_hire_fixed_fm_submission_tabdpt_hire_fixed_testpreds.npy — 本NBでは学習していない（GPU必須のため）
[2026-08-16 14:16:23] [INFO] 【再利用】tabdpt(hire_fixed): val=20260816_68_foundation_models_3way_tabdpt_hire_fixed_valpreds.npy / test=20260816_70_hire_fixed_fm_submission_tabdpt_hire_fixed_testpreds.npy — 本NBでは学習していない（GPU必須のため）


INFO:71_model_zoo_cpu:【再利用】tabdpt(hire_fixed): val=20260816_68_foundation_models_3way_tabdpt_hire_fixed_valpreds.npy / test=20260816_70_hire_fixed_fm_submission_tabdpt_hire_fixed_testpreds.npy — 本NBでは学習していない（GPU必須のため）


  ⚠️ 68_のval予測は速度優先の縮小設定(TabPFN n_estimators=4)。絶対値の比較には使わず相関のみ見る
  ⚠️ 70_のtest予測は本番設定（Public 0.511832 を出した構成の素材）
【再利用】poolC: 20260816_pool_poolC_weighted.csv — 50_/51_/53_/61_×2/62_×4 の full441 weighted 8本平均。本NBでは AutoGluon を一切実行していない。64_で追加中のシードは未反映
[2026-08-16 14:16:24] [INFO] 【再利用】poolC: 20260816_pool_poolC_weighted.csv — 50_/51_/53_/61_×2/62_×4 の full441 weighted 8本平均。本NBでは AutoGluon を一切実行していない。64_で追加中のシードは未反映


INFO:71_model_zoo_cpu:【再利用】poolC: 20260816_pool_poolC_weighted.csv — 50_/51_/53_/61_×2/62_×4 の full441 weighted 8本平均。本NBでは AutoGluon を一切実行していない。64_で追加中のシードは未反映


【再利用】54_ CatBoost最良 val予測: 20260815_54_l2_m_interaction_R0_memofix_plus_LM_valpreds.npy（8シード平均） — 本NBのCatBoost基準との照合用。相関 0.9777, val 0.505477 vs 本NB 0.516147
[2026-08-16 14:16:24] [INFO] 【再利用】54_ CatBoost最良 val予測: 20260815_54_l2_m_interaction_R0_memofix_plus_LM_valpreds.npy（8シード平均） — 本NBのCatBoost基準との照合用。相関 0.9777, val 0.505477 vs 本NB 0.516147


INFO:71_model_zoo_cpu:【再利用】54_ CatBoost最良 val予測: 20260815_54_l2_m_interaction_R0_memofix_plus_LM_valpreds.npy（8シード平均） — 本NBのCatBoost基準との照合用。相関 0.9777, val 0.505477 vs 本NB 0.516147


## 16. 相関行列とゲート判定

In [24]:
# ---- val（535名）での相関行列 ----
valmat = {k: v["val"] for k, v in ZOO.items()}
valmat.update({k: v["val"] for k, v in REUSED.items() if v["val"] is not None})
C = pd.DataFrame({a: {b: np.corrcoef(valmat[a], valmat[b])[0, 1] for b in valmat} for a in valmat})
print("=== val予測の相関行列（535名） ===")
print(C.round(3).to_string())

# ---- ゲート判定 ----
rows = []
for k, d in ZOO.items():
    if k == "catboost":
        continue
    vl = log_loss(y_val, d["val"]); cr = np.corrcoef(d["val"], cb_val)[0, 1]
    a, b = vl <= CB_VAL + GATE_VAL_MARGIN, cr < GATE_CORR_MAX
    rows.append(dict(model=k, view=d["view"], source=d["source"], val=round(vl, 6),
                     vs_cb=round(vl - CB_VAL, 6), seed_sd=(np.nan if np.isnan(d["sd"]) else round(d["sd"], 5)),
                     corr_cb=round(cr, 4), gateA=a, gateB=b, 採用候補=a and b))
R = pd.DataFrame(rows).sort_values("val")
print(f"\n=== ゲート判定（CatBoost val = {CB_VAL:.6f}）===")
print(f"(A) val ≤ {CB_VAL + GATE_VAL_MARGIN:.6f} / (B) corr < {GATE_CORR_MAX}\n")
print(R.to_string(index=False))
R.to_csv(OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_gate.csv", index=False)

ok = R[R["採用候補"]]["model"].tolist()
print(f"\n>>> 採用候補: {ok if ok else 'なし'}")


=== val予測の相関行列（535名） ===
                    catboost    ebm     gp  svm_rbf    mlp  extratrees  logreg  tabpfn(hire_fixed)  tabicl(hire_fixed)  tabdpt(hire_fixed)
catboost               1.000  0.936  0.847    0.817  0.836       0.916   0.858               0.877               0.887               0.834
ebm                    0.936  1.000  0.805    0.766  0.814       0.850   0.843               0.869               0.873               0.783
gp                     0.847  0.805  1.000    0.923  0.820       0.836   0.858               0.725               0.768               0.888
svm_rbf                0.817  0.766  0.923    1.000  0.797       0.837   0.851               0.748               0.764               0.914
mlp                    0.836  0.814  0.820    0.797  1.000       0.832   0.959               0.750               0.835               0.854
extratrees             0.916  0.850  0.836    0.837  0.832       1.000   0.853               0.818               0.865               0.866
lo

In [25]:
# ---- Test予測（2502名）での相関。プールCとの距離が実際のアンサンブル価値に効く ----
testmat = {k: v["test"] for k, v in ZOO.items()}
testmat.update({k: v["test"] for k, v in REUSED.items() if v["test"] is not None})
CT = pd.DataFrame({a: {b: np.corrcoef(testmat[a], testmat[b])[0, 1] for b in testmat} for a in testmat})
print("=== Test予測の相関行列（2502名） ===")
print(CT.round(3).to_string())

if poolC is not None:
    print(f"\n=== プールCとの距離（ノイズ床 MAD 0.02122 / AutoGluonシード間MAD 0.02381）===")
    for k, v in testmat.items():
        if k.startswith("poolC"): continue
        print(f"  {k:26s} corr={np.corrcoef(v, poolC)[0,1]:.4f}  MAD={np.abs(v-poolC).mean():.5f}")
    print("\n  MAD が 0.02381 を大きく超えるものほど『本当に別のモデル』"
          "（[[refit-chaos-noise-floor]]）。相関だけでなくこちらも見る。")


=== Test予測の相関行列（2502名） ===
                      catboost    ebm     gp  svm_rbf    mlp  extratrees  logreg  tabpfn(hire_fixed)  tabicl(hire_fixed)  tabdpt(hire_fixed)  poolC(AutoGluon8本平均)
catboost                 1.000  0.947  0.813    0.788  0.819       0.893   0.843               0.907               0.916               0.804                 0.974
ebm                      0.947  1.000  0.791    0.749  0.818       0.855   0.842               0.887               0.893               0.770                 0.932
gp                       0.813  0.791  1.000    0.928  0.815       0.837   0.847               0.712               0.784               0.875                 0.787
svm_rbf                  0.788  0.749  0.928    1.000  0.787       0.846   0.826               0.732               0.791               0.901                 0.779
mlp                      0.819  0.818  0.815    0.787  1.000       0.852   0.969               0.758               0.844               0.876                 0

## 17. まとめと次のアクション

### 読み方（事前に決めた運用）

- **`採用候補 = True` のモデルだけ**を次のブレンド検討に進める。
  val の順位付けはしない（[[validation-asymmetry]]: 「改善」方向の的中は通算5/11）。
- **`vs_cb` が `NOISE_HINT`(0.003) 以内の差は読まない。**
- Test相関行列で **プールCとの MAD が 0.02381 を超えている**ものが、実際に情報を足せる候補。

### 本ノートブックでやらないこと（意図的）

- **提出ファイルを作らない。** ブレンド重みの探索もしない。
  重みを学習すると [[ensemble-oof-overfitting]] の罠に入るので、
  採用が決まったら `63_`/`66_` と同じ**固定重み**で別ノートブックにて作る。
- **CatBoost の再探索をしない**（[[hyperparameter-retuning-exhausted]] で打ち止め）。

### Public の使い方（10回/日）

```
0回   … ここで出た新モデルの単体提出（単体スコアは決定に使わないので無駄）
1-2回 … 合格した partner を加えた新ブレンド
残り  … プールCの独立ドロー追加（第94節で3→8本 -0.0013 実証済み）
```

### 想定される結末

`36_` が441列で却下した SVM・MLP が低次元ビューで蘇れば、**GPU無しで手に入る新しい partner**になる。
逆に全滅した場合も、「ビューを変えても駄目だった」という形で `36_` の結論を正しい条件で確定でき、
CPU側のモデル探索を閉じられる。どちらに転んでも次の判断材料になる。

---
# 18. 提出ファイルの生成（EBM単体）— スコアではなく「情報」を買う提出

## なぜ EBM 単体なのか

第16-17節の通り、`71_` で提出に値するのは **EBM だけ**（他5モデルは `argmin w = 1.0`
＝どんな重みで混ぜても CatBoost 単体に勝てない）。その EBM も
`CatBoost+TabPFN` に足すと単調に悪化するので、**ブレンドとしては不採用**である。

それでも単体を1回出す価値があるのは、**ブレンドでは結果が読めないから**:

| 構成 | 現最良とのMAD | 判定 |
|---|---|---|
| 現最良×0.90 + EBM×0.10 | 0.00720 | ノイズ床 0.02122 未満 → **読めない** |
| 現最良×0.85 + EBM×0.15 | 0.01081 | 同上 → **読めない** |
| **EBM 単体** | **0.07205** | ノイズ床の3.4倍 → **必ず読める** |

読める差にするには EBM の重みを 0.30 近くまで上げる必要があるが、val はその領域で単調悪化と言う。
**「読める結果」と「良い結果」は同時に買えない**ので、情報を取りにいくなら単体になる。

## 事前登録: val→Public のギャップは「分割の性質」か「モデル固有」か

535名の検証セットは分解能 ±0.0099 で、**このプロジェクトが当ててきた改善8件すべてが分解能未満**
だった（＝判別器としては機能していない）。だが **val→Public のギャップが安定した加算定数**なら、
判別器としてはダメでも**変換器としては使える**。そうなれば「val を測って Public を予測する」ができ、
測定のために提出枠を燃やす必要がなくなる。

```
54_ CatBoost : val 0.505477 → Public 0.515030   ギャップ +0.009553
EBM          : val 0.514536 → Public ???
```

**仮説A（ギャップは分割の性質）が正しければ EBM の Public ≒ 0.524089。**

| 実測 | 結論 |
|---|---|
| **0.5191 〜 0.5291**（±0.005） | 仮説A支持。val を**変換器**として使える |
| それ以外 | ギャップはモデル固有。val は変換器としても使えない → **2,761名OOFへの移行が必須**（`72_`で比較材料が揃う） |

CatBoost と EBM は木の構造も正則化も全く違うので、**2点が一致すれば偶然ではない**と言える。
外れた場合も「535名検証を捨てる」判断の決定打になる。どちらに転んでも次が決まる。

> ⚠️ **これは新最良を狙う提出ではない。** 予想 0.524 に対し現最良は 0.508699。
> 順位を落としたくない事情があるなら、この提出は見送ってよい。

In [ ]:
# ============================================================
# EBM単体の提出ファイルを作る
#   ※ ここで使う ZOO["ebm"]["test"] は第14節で作った Test予測（全件学習・3シード平均）。
#      キャッシュから読み込まれている場合は source 列が「キャッシュ再利用」になっている。
# ============================================================
SUBMIT_MODEL = "ebm"

if SUBMIT_MODEL not in ZOO:
    print(f"⚠️ {SUBMIT_MODEL} が ZOO に無いのでスキップ（SKIP_MODELS か interpret 未導入）")
else:
    pred = ZOO[SUBMIT_MODEL]["test"]
    print(f"元データ: ZOO['{SUBMIT_MODEL}']  view={ZOO[SUBMIT_MODEL]['view']}  "
          f"source={ZOO[SUBMIT_MODEL]['source']}")

    # 提出は sample_submission の社員ID順・ヘッダー無し2列（本コンペの形式）
    sample = pd.read_csv(PROJECT_ROOT/"data"/"input"/"sample_submission.csv",
                         header=None, names=[ID_COL, "_見本"])
    # test_features_full は set_index(社員ID) 済みなので index が社員ID
    pred_s = pd.Series(pred, index=test_features_full.index)
    assert set(sample[ID_COL]) == set(pred_s.index), "社員IDの集合が sample_submission と一致しない"
    submission = pd.DataFrame({ID_COL: sample[ID_COL],
                               "p": pred_s.loc[sample[ID_COL]].to_numpy()})

    # 健全性チェック（0/1に張り付いていないか、行数、欠損）
    assert len(submission) == 2502, len(submission)
    assert submission["p"].notna().all(), "欠損がある"
    assert submission["p"].between(1e-6, 1 - 1e-6).all(), "確率が0または1に張り付いている"

    sub_path = OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_{SUBMIT_MODEL}_standalone.csv"
    submission.to_csv(sub_path, index=False, header=False)   # ヘッダー無し
    logger.info(f"提出ファイル作成: {sub_path}")

    print(f"\n提出ファイル: {sub_path}")
    print(f"  行数 {len(submission)} / 平均 {submission['p'].mean():.4f} / "
          f"範囲 [{submission['p'].min():.4f}, {submission['p'].max():.4f}]")
    print(submission.head(3).to_string(index=False, header=False))


In [ ]:
# ============================================================
# 事前登録した判定窓を、実際の数値で印字しておく（提出前に確定させる）
#   【再利用】54_ の Public 0.515030 は過去の提出結果。本NBで計算したものではない。
# ============================================================
VAL_CB54, PUB_CB54 = 0.505477, 0.515030      # 54_ R0_memofix_plus_LM（単層CatBoost最良）
GAP = PUB_CB54 - VAL_CB54

if SUBMIT_MODEL in ZOO:
    val_ebm = log_loss(y_val, ZOO[SUBMIT_MODEL]["val"])
    pred_pub = val_ebm + GAP
    print("=== 事前登録した判定（提出前に確定） ===")
    print(f"  54_ CatBoost : val {VAL_CB54:.6f} → Public {PUB_CB54:.6f}   ギャップ {GAP:+.6f}")
    print(f"  {SUBMIT_MODEL:12s} : val {val_ebm:.6f} → Public ???")
    print(f"\n  仮説A（ギャップは分割の性質）の予測: Public ≒ {pred_pub:.6f}")
    print(f"  判定窓: {pred_pub-0.005:.4f} 〜 {pred_pub+0.005:.4f}")
    print( "    窓内 → val を『変換器』として使える（判別器としては引き続き使えない）")
    print( "    窓外 → ギャップはモデル固有。535名検証は変換器としても不可 → 2,761名OOFへ移行")

    # 読めるかどうかの確認（ノイズ床との比較）
    _best = _find("*/*_pool_top150_hire_fixed_avg.csv")
    if _best is not None:
        b = pd.read_csv(_best, header=None, names=[ID_COL, "p"]).set_index(ID_COL)
        b = b.loc[test_features_full.index, "p"].to_numpy()
        mad = np.abs(ZOO[SUBMIT_MODEL]["test"] - b).mean()
        print(f"\n  現最良({_best.name})とのMAD = {mad:.5f} "
              f"（ノイズ床0.02122の{mad/0.02122:.1f}倍）→ {'必ず読める' if mad>0.02122 else '読めない'}")


## 18.1 提出後にやること

1. Public スコアを上の判定窓と突き合わせ、**仮説Aの採否を先に決める**（スコアの良し悪しではなく）
2. `submit_result_report.md` に追記し、[[validation-asymmetry]] のデータ点として記録する
3. 窓外だった場合は `72_` の2プロトコル比較（535名 chronological vs 2,761名 OOF）と合わせて、
   **検証設計そのものを差し替える**判断に進む

**この提出でスコアが悪くても、それは失敗ではない**（予想0.524、現最良0.508699）。
買っているのは順位ではなく「検証セットを今後どう使うか」の決着である。